In [1]:
import torch
from torch_geometric.data import Data
from torch_geometric.nn import MessagePassing

In [5]:
# Function to create a bifurcating tree
def create_bifurcating_tree(depth):
    edges = []
    for i in range(2**depth - 1):
        if 2 * i + 1 < 2**depth - 1:
            edges.append([i, 2 * i + 1])
            edges.append([i, 2 * i + 2])
    return torch.tensor(edges, dtype=torch.long).t().contiguous()

class CountLeaves(MessagePassing):
    def __init__(self):
        # Note that we use the `target_to_source` flow direction because our tree has the graph laid out from root to leaves.
        super(CountLeaves, self).__init__(aggr='add', flow='target_to_source')

    def forward(self, x, edge_index):
        # Forward is how we run the whole model.
        num_nodes = x.size(0)
        return self.propagate(edge_index, size=(num_nodes, num_nodes), x=x)

    def message(self, x_j):
        # Pass the feature of the source node (x_j) as the message.
        return x_j


depth=3
edge_index = create_bifurcating_tree(depth)

# Define initial messages: 1 for leaves, 0 for non-leaves
num_nodes = 2**depth - 1  # Total number of nodes in the tree
x = torch.zeros(num_nodes, 1)
x[2**(depth-1)-1:] = 1  # Set 1 for leaf nodes

# Create data object
data = Data(x=x, edge_index=edge_index)

# Initialize and run the CountLeaves model
model = CountLeaves()
leaf_count = model(data.x, data.edge_index)
for _ in range(depth-2):
    data.x = leaf_count
    leaf_count = model(data.x, data.edge_index)

print("Leaf count at root node:", leaf_count[0])

Leaf count at root node: tensor([4.])
